# Feature Engineering

In [81]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import database

## Import raw feature data from the db

<b style="color:DodgerBlue;">Only fetches everything until the last complete month</b>

In [82]:
data = database.db_requester_fetchall(sql_script="all_MonthlyFeatures.sql")

In [83]:
arr_raw = np.array(data)
arr_raw[np.where(arr_raw == None)[0]]
# np.isnan(arr_raw)

array([['UNRATE', '2025-10-01', None, '2026-08-10 22:57:41']],
      dtype=object)

## Data prep
- Impute missing values

In [84]:
df_raw = pd.DataFrame(data, columns=['series_id', 'month_year', 'value', 'timestamp'])
df_raw['month_year'] = pd.to_datetime(df_raw['month_year'])
df_raw

,series_id,month_year,value,timestamp
0,ICSA,2000-12-01,1.730000e+06,2026-08-10 22:57:41
1,INDPRO,2000-12-01,9.234150e+01,2026-08-10 22:57:41
2,JTSJOL,2000-12-01,5.088000e+03,2026-08-10 22:57:41
3,PAYEMS,2000-12-01,1.327160e+05,2026-08-10 22:57:41
4,UNRATE,2000-12-01,3.900000e+00,2026-08-10 22:57:41
...,...,...,...,...
1530,ICSA,2026-06-01,8.900000e+05,2026-08-10 22:57:41
1531,INDPRO,2026-06-01,1.026395e+02,2026-08-10 22:57:41
1532,JTSJOL,2026-06-01,7.359000e+03,2026-08-10 22:57:41
1533,PAYEMS,2026-06-01,1.588810e+05,2026-08-10 22:57:41


In [106]:
df = df_raw.pivot(
    index='month_year',
    columns='series_id',
    values='value'
).reset_index()
df['month_year'] = df['month_year'].dt.to_period('M')
df.columns.name = None

df

,month_year,ICSA,INDPRO,JTSJOL,PAYEMS,UNRATE
0,2000-12,1730000.0,92.3415,5088.0,132716.0,3.9
1,2001-01,1360000.0,91.9020,5234.0,132703.0,4.2
2,2001-02,1485000.0,91.3034,5097.0,132788.0,4.2
3,2001-03,1936000.0,91.1162,4762.0,132751.0,4.3
4,2001-04,1587000.0,90.7891,4615.0,132458.0,4.4
...,...,...,...,...,...,...
302,2026-02,863000.0,101.9263,6922.0,158436.0,4.4
303,2026-03,832000.0,101.6172,6887.0,158650.0,4.3
304,2026-04,831000.0,102.4196,7585.0,158798.0,4.3
305,2026-05,1058000.0,102.5606,7537.0,158861.0,4.3


<b style="color:DodgerBlue;">IMPORTANT: Do not forget to treat missing values</b>

In [145]:
df[df.isna().any(axis=1)]

,month_year,ICSA,INDPRO,JTSJOL,PAYEMS,UNRATE
298,2025-10,907000.0,101.2195,7170.0,158408.0,NaN


In [149]:
features = ['ICSA', 'INDPRO', 'JTSJOL', 'PAYEMS', 'UNRATE']
df_imputed = df.copy()
df_imputed[features] = df[features].interpolate(method="linear")
df_imputed[df_imputed.isna().any(axis=1)]
# df_imputed

,month_year,ICSA,INDPRO,JTSJOL,PAYEMS,UNRATE


## Feature sets, plot, and correlation analysis
- Time lagged `UNRATE`
- Normalize data
- Try converting data to a common unit e.g. rate of change

In [157]:
df_imputed[features].corr()

,ICSA,INDPRO,JTSJOL,PAYEMS,UNRATE
ICSA,1.000000,-0.427989,-0.188280,-0.284911,0.564504
INDPRO,-0.427989,1.000000,0.613091,0.715330,-0.617404
JTSJOL,-0.188280,0.613091,1.000000,0.840835,-0.605025
PAYEMS,-0.284911,0.715330,0.840835,1.000000,-0.652774
UNRATE,0.564504,-0.617404,-0.605025,-0.652774,1.000000


Consider following models:
- Ridge regression -> L2 helps with multicollinearity
- Regression tree
- Random forest